# MCP Incident Detection, Containment, and Forensics

Investigate a synthetic cross-tenant MCP incident from admitted telemetry through verified containment. The central rule is **evidence is not inference, control state is not effect proof, and model text is not authority**.

## Safety and scope

Everything is deterministic, synthetic, and offline. No production server, credential, network, SIEM, or control plane is contacted. The notebook imports the canonical `lab.py` rather than duplicating its security implementation.

## Objectives

You will admit signed evidence, correlate independent effects, preserve hypotheses as untrusted proposals, scope with tenant authority, create a redacted forensic package, execute an exact signed plan, reconcile an unknown outcome across restart, and prove containment.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import runpy

lab = runpy.run_path(Path("lab.py"))
environment = lab["build_environment"]()
tenants = frozenset({"acme", "beta"})
(len(environment.store.events), lab["PROTOCOL_VERSION"], lab["SCHEMA_VERSION"])

## 1. Admit evidence before reasoning

The store accepts only known emitters, authorized tenants, valid UTC timing, valid content signatures, and a contiguous per-emitter hash chain. Exact duplicates are idempotent; conflicting copies are rejected.

In [ ]:
[(event.event_id, event.emitter_id, event.kind.value, event.outcome.value)
 for event in environment.malicious_events]

In [ ]:
first_event = environment.malicious_events[0]
duplicate_result = environment.store.ingest(first_event)
assert duplicate_result.status == lab["IngestStatus"].DUPLICATE
duplicate_result

In [ ]:
tampered = replace(first_event, action="deployment.delete")
tamper_store = lab["EvidenceStore"]()
tamper_result = tamper_store.ingest(tampered)
assert tamper_result.reason_code == "INTEGRITY_INVALID"
tamper_result

## 2. Detect and correlate

A finding is a rule result with a version and exact evidence IDs. A critical alert requires multiple high-confidence findings and independent protected-resource evidence; unexpected egress alone is insufficient.

In [ ]:
findings = environment.detector.detect(environment.malicious_events)
[(item.kind.value, item.rule_id, item.severity.value, item.evidence_ids)
 for item in findings]

In [ ]:
benign_findings = environment.detector.detect(environment.benign_events)
blocked_export_findings = environment.detector.detect(environment.safe_denial_events)
assert environment.detector.alert(benign_findings, now=lab["NOW"]) is None
assert environment.detector.alert(blocked_export_findings, now=lab["NOW"]) is None
([item.kind.value for item in benign_findings],
 [item.kind.value for item in blocked_export_findings])

In [ ]:
alert = environment.detector.alert(findings, now=lab["NOW"])
assert alert is not None
(alert.alert_id, alert.severity.value, alert.evidence_ids, alert.alert_digest)

## 3. Keep hypotheses separate from facts

The alert opens a case. A model may propose a hypothesis only against admitted case evidence. It cannot mark that hypothesis as fact or declare the incident.

In [ ]:
repository = lab["IncidentRepository"]()
manager = lab["IncidentManager"](environment.store, repository)
case = manager.open_from_alert(
    alert, analyst_id="analyst-7", allowed_tenants=tenants
)
(case.incident_id, case.state.value, case.version)

In [ ]:
case = manager.add_hypothesis(
    case,
    proposed_by="assistant-model",
    text="credential theft may explain the observed sequence",
    evidence_ids=alert.evidence_ids,
)
assert case.hypotheses[-1].accepted_as_fact is False
case.hypotheses[-1]

In [ ]:
case = manager.declare(
    case,
    findings,
    responder_id="responder-2",
    responder_roles=frozenset({"incident_responder"}),
    now=lab["NOW"] + timedelta(seconds=1),
)
(case.state.value, case.declared_at, case.version)

## 4. Scope only inside investigator authority

Graph joins connect exact trace, operation, artifact, workload, session, and credential identifiers. Authorization occurs before retrieval and graph expansion; reachability is neither identity nor permission.

In [ ]:
restricted_report = lab["ScopeEngine"]().scope(
    case,
    environment.store,
    analyst_tenants=frozenset({"acme"}),
    started_at=lab["NOW"] - timedelta(minutes=2),
    ended_at=lab["NOW"],
)
assert "beta" not in restricted_report.confirmed_tenants
assert restricted_report.excluded_unauthorized_count == 2
(restricted_report.confirmed_tenants,
 restricted_report.excluded_unauthorized_count)

In [ ]:
report = lab["ScopeEngine"]().scope(
    case,
    environment.store,
    analyst_tenants=tenants,
    started_at=lab["NOW"] - timedelta(minutes=2),
    ended_at=lab["NOW"],
)
(report.confirmed_tenants, report.workloads, report.credentials,
 report.destinations,
 report.excluded_unauthorized_count)

In [ ]:
scope_metrics = lab["evaluate_scope"](
    report, environment.actual_incident_evidence
)
assert scope_metrics.precision == scope_metrics.recall == 1.0
scope_metrics

## 5. Preserve a redacted, verifiable package

The package contains pseudonyms and digests instead of raw principals, sessions, credentials, resources, or destinations. Its canonical manifest and custody records are integrity protected.

In [ ]:
vault = lab["ForensicVault"]()
package = vault.preserve(
    case,
    report,
    environment.store,
    actor_id="collector-1",
    actor_roles=frozenset({"forensic_collector"}),
    allowed_tenants=tenants,
    now=lab["NOW"] + timedelta(seconds=2),
    retention_until=lab["NOW"] + timedelta(days=365),
    legal_hold=True,
)
assert vault.verify(package)
(package.package_id, package.manifest_digest, len(package.evidence),
 package.legal_hold)

In [ ]:
sample = package.evidence[0]
(sample.tenant_pseudonym, sample.principal_pseudonym,
 sample.resource_digest, sample.destination_digest, sample.source_locator)

In [ ]:
invalid_package = replace(package, signature="tampered")
assert vault.verify(invalid_package) is False
vault.verify(invalid_package)

In [ ]:
package = vault.transfer(
    package,
    actor_id="custody-officer",
    actor_roles=frozenset({"forensic_custodian"}),
    new_custodian="legal-vault",
    reason="legal-hold transfer",
    now=lab["NOW"] + timedelta(seconds=3),
)
assert vault.verify(package)
[(record.sequence, record.action, record.reason) for record in package.custody]

## 6. Bind approval to one exact containment plan

The plan covers every scoped binding, artifact, workload, credential, session, and unapproved egress destination. Approval signs its canonical digest, tenant, incident, policy, role, and expiry.

In [ ]:
plan = lab["make_plan"](
    case, report, requested_by="responder-2",
    now=lab["NOW"] + timedelta(seconds=3),
)
[(action.kind.value, action.target, action.secondary)
 for action in plan.actions]

In [ ]:
receipt = lab["approve_plan"](
    plan,
    approver_id="commander-1",
    approver_role="incident_commander",
    now=lab["NOW"] + timedelta(seconds=4),
)
assert lab["verify_plan"](plan) and lab["verify_receipt"](receipt)
(receipt.receipt_id, receipt.plan_digest, receipt.expires_at)

## 7. Treat a timeout as outcome unknown

The control mutation may commit before the response is lost. Keep the logical operation ID and request digest stable, persist the journal, inspect target state, and reconcile before retrying.

In [ ]:
executor = lab["ContainmentExecutor"]()
first_execution = executor.execute(
    "containment-operation-1", plan, receipt, package, vault, case,
    now=lab["NOW"] + timedelta(seconds=5),
    simulate_lost_response=True,
)
assert first_execution.status == lab["ExecutionStatus"].UNKNOWN
(first_execution.status.value, first_execution.reason_code,
 executor.controls.revision)

In [ ]:
restarted = lab["ContainmentExecutor"](
    executor.controls, executor.journal
)
reconciled = restarted.execute(
    "containment-operation-1", plan, receipt, package, vault, case,
    now=lab["NOW"] + timedelta(seconds=6),
)
assert reconciled.status == lab["ExecutionStatus"].RECONCILED
assert restarted.controls.revision == 1
(reconciled.status.value, reconciled.reason_code, restarted.controls.revision)

## 8. Prove containment from controls and effects

All required controls must be active, the evidence package must still verify, and no affected identity or destination may produce a completed post-cut effect.

In [ ]:
proof = lab["prove_containment"](
    plan, restarted.controls, package, vault, environment.store,
    applied_at=reconciled.applied_at,
)
assert proof.complete
proof

In [ ]:
case = manager.mark_contained(
    case,
    proof,
    reconciled,
    responder_roles=frozenset({"incident_responder"}),
)
assert case.state == lab["CaseState"].CONTAINED
(case.state.value, case.version)

### Adversarial check: a late effect defeats the claim

Run a separate complete scenario, then admit an independently signed post-cut completion from a contained workload. Control state still looks correct, but proof must fail.

In [ ]:
late_environment = lab["build_environment"]()
late_findings = late_environment.detector.detect(late_environment.malicious_events)
late_alert = late_environment.detector.alert(late_findings, now=lab["NOW"])
late_repository = lab["IncidentRepository"]()
late_manager = lab["IncidentManager"](late_environment.store, late_repository)
late_case = late_manager.open_from_alert(
    late_alert, analyst_id="analyst-7", allowed_tenants=tenants
)
late_case = late_manager.declare(
    late_case, late_findings,
    responder_id="responder-2",
    responder_roles=frozenset({"incident_responder"}),
    now=lab["NOW"] + timedelta(seconds=1),
)
late_report = lab["ScopeEngine"]().scope(
    late_case, late_environment.store, analyst_tenants=tenants,
    started_at=lab["NOW"] - timedelta(minutes=2), ended_at=lab["NOW"],
)
late_vault = lab["ForensicVault"]()
late_package = late_vault.preserve(
    late_case, late_report, late_environment.store,
    actor_id="collector-1",
    actor_roles=frozenset({"forensic_collector"}),
    allowed_tenants=tenants,
    now=lab["NOW"] + timedelta(seconds=2),
    retention_until=lab["NOW"] + timedelta(days=365),
    legal_hold=True,
)
late_plan = lab["make_plan"](
    late_case, late_report, requested_by="responder-2",
    now=lab["NOW"] + timedelta(seconds=3),
)
late_receipt = lab["approve_plan"](
    late_plan, approver_id="commander-1",
    approver_role="incident_commander",
    now=lab["NOW"] + timedelta(seconds=4),
)
late_executor = lab["ContainmentExecutor"]()
late_execution = late_executor.execute(
    "late-check-operation", late_plan, late_receipt,
    late_package, late_vault, late_case,
    now=lab["NOW"] + timedelta(seconds=5),
)
template = lab["EventTemplate"](
    event_id="event-late-effect",
    occurred_at=lab["NOW"] + timedelta(seconds=6),
    observed_at=lab["NOW"] + timedelta(seconds=7),
    tenant_id="acme",
    resource_tenant="acme",
    principal_id="user:acme:test",
    server_id=late_case.server_id,
    artifact_digest=late_report.artifacts[0],
    workload_id=late_report.workloads[0],
    trace_id="e" * 32,
    operation_id="operation-late",
    session_id=late_report.sessions[0],
    credential_id=late_report.credentials[0],
    kind=lab["EventKind"].CONTROL,
    action="customer.read",
    resource="customer://acme/late",
    destination="collect.invalid",
    policy_decision="deny",
    outcome=lab["Outcome"].COMPLETED,
    source_locator="vault://late/effect",
)
late_event = lab["EventEmitter"](
    "control.audit", lab["EVENT_KEYS"]["control.audit"]
).emit(template)
late_environment.store.ingest(
    late_event, now=lab["NOW"] + timedelta(seconds=7)
)
failed_proof = lab["prove_containment"](
    late_plan, late_executor.controls, late_package, late_vault,
    late_environment.store, applied_at=late_execution.applied_at,
)
assert not failed_proof.complete
failed_proof

## 9. Evaluate the response

Use labelled cases and declared denominators. Detection recall alone rewards blocking everything; pair it with precision and valid-service disruption.

In [ ]:
demo = lab["run_demo"]()
{
    "detection_precision": demo["detection_metrics"].precision,
    "detection_recall": demo["detection_metrics"].recall,
    "scope_precision": demo["scope_metrics"].precision,
    "scope_recall": demo["scope_metrics"].recall,
    "evidence_completeness": demo["incident_metrics"].evidence_completeness,
    "time_to_detect_seconds": demo["incident_metrics"].time_to_detect_seconds,
    "time_to_contain_seconds": demo["incident_metrics"].time_to_contain_seconds,
    "valid_service_disruption_rate": demo["incident_metrics"].valid_service_disruption_rate,
}

In [ ]:
demo["otel"]

## Exercises

1. Remove the protected-resource audit event. Explain why a finding may remain while declaration fails.
2. Add a bounded buffer for authentically signed but reordered events; never silently skip a chain gap.
3. Design an emergency quarantine receipt that is narrower and shorter-lived than normal containment approval.
4. Map one event to OpenTelemetry or OCSF and document every field transformation.
5. Add a delayed event whose occurrence time is before the cut but observed time is after it. State which clock and uncertainty policy controls the verdict.
6. Produce the recovery handoff for Course 22 without implementing rollback here.

## Reflection

- Which source in your environment independently proves protected-resource effects?
- What evidence would be lost if you isolated a workload immediately?
- Which cross-tenant scope expansion requires additional authority?
- How does your control plane distinguish no response from no effect?
- What fact would falsify your current containment claim?